# Diseña y visualiza con D3.js datos de siniestralidad en carretera

## 0. Introducción

Los datos de siniestralidad en carretera representan **una de las grandes preocupaciones** tanto institucionales como sociales, datos que siempre apelan tanto a la responsabilidad individual como colectiva, y que gracias a la Direccion General de Tráfico y su participación a través del portal de datos abiertos Datos.gob.es se pueden analizar y visualizar de forma intuitiva.

A partir de los datos disponibles para el año 2024 es posible **extraer métricas cuantitativas** para hacernos una idea de las situaciones y consecuencias de los accidentes de tráfico a lo largo del tiempo.

En la base de datos encontramos más de cien mil accidentes **agregados por días de la semana**, con información respecto al número total de víctimas (fallecidos, heridos graves o heridos leves en horizontes temporales tanto de 24h como de 30 días) y respecto al **tipo de accidente, tipo de vía, titularidad de la vía y municipio**.

En este ejericio realizaremos una serie de operaciones sencillas utilizando Python para poder extraer las métricas de nuestro interés en formato JSON, para su posterior visualización en Javascript, en concreto en D3.js.

El objetivo de este ejercicio es la **visualización de varias de las variables** que caracterizan la siniestralidad en carretera. Para ello el ejericio se estructura en tres pasos en esta etapa a desarrollar en Python:

1.   **Lectura** del fichero de datos de la DGT
   *    Creación de un **Dataframe**
   *    Creacion de una **fecha para cada entrada**


2.   **Cálculo** de las métricas para caracterizar los datos disponibles
*    Suma del **total de víctimas** en cada hora del día
*    Suma de **accidentes por categoría**
*    Asignación de valores a los **municipios**

3.   **Exportar** las métricas en formato JSON

1. Lectura del fichero de datos
1.1 Creación de un Dataframe
Para la lectura de los datos utilizaremos la libreria Pandas de Python, que nos permite estructurar los datos en filas y columnas de forma intuitiva y operar sobre ellos.

Igualmente, y dado que tenemos la fecha en formato de año, mes y día de la semana así como la hora del accidente, necesitaremos una librería para modificar y crear formatos de fecha que podamos manejar fácilmente. Para ello disponemos de la librería Datetime en Python.

La libreria IO y files de Google Colab nos permite hacer una lectura de datos desde otras localizaciones fuera de este notebook

Por último para la parte geográfica de asignación de valores a los diferentes municipios necesitaremos una librería para poder procesar geometrías en forma de polígonos, y lo haremos a través de Geopandas.

In [ ]:
import pandas as pd
import numpy as np
import datetime
import geopandas as gpd
import warnings
warnings.filterwarnings("ignore")

Procedemos entonces a la lectura del fichero de entrada en forma de pandas a partir de un fichero en formato .csv. Es importante utilizar el encoding del texto adecuado para una correcta interpretacion de determinados caracteres. En este caso, 'latin-1' es el adecuado dado que los datos que contiene están mayormente en castellano y asi la ortografía latina será bien interpretada por Python.
Una vez importados los datos y almacenados en un dataframe podemos ver el nombre de las variables y su tipo a través del comando `.info()` de Pandas, que nos ofrece:


*   **Número de entradas** en el dataframe
*   **Nombre de las columnas** asociada con cada variable
*   **Tipo de cada variable**, incluyendo string, float o int.

In [ ]:
content = pd.read_csv(https://github.com/Admindatosgobes/Laboratorio-de-Datos/blob/main/Visualizaciones/Dise%C3%B1a%20y%20visualiza%20con%20D3.js%20datos%20de%20siniestralidad%20en%20carretera/TABLA_ACCIDENTES_24.csv, encoding='latin-1')


In [ ]:
content.info()

Una vez ejecutada la función `.info()` podemos empezar a operar sobre la estructura y las variables contenidas en el dataframe.

### 1.2 Creación de una fecha para cada entrada

La información temporal de los datos viene en **cuatro columnas separadas** para el año, el mes, el día de la semana (de lunes a domingo), y por último la hora del accidente.
Dado que los datos está agregados en origen por día de la semana y no por día natural, intentaremos crear una serie temporal desde una perspectiva semanal.
Para ello son necesarias una serie de transformaciones, en las cuales **crearemos primero una columna con la fecha y hora** de cada accidente de forma más compacta que cuatro columnas por separado.
En primer lugar, reasignamos el nombre de cada columna a los estándares de la librería Datetime de Python.

In [ ]:
content = content.rename(columns={'ANYO': 'year', 'MES': 'month', 'DIA_SEMANA': 'day','HORA':'hour'})

Como realizaremos un analisis semanal de la ocurrencia de accidentes, asignamos valores constantes tanto al año como al mes, ya que queremos ver la variabilidad semanal y no estacional. De esta forma podremos superponer diferentes días de la semana sobre la misma escala temporal de mes y año.
Para ello usaremos la función lambda que opera sobre todos los elementos de una columna de un dataframe, asignándoles el valor que queramos.

In [ ]:
content['year']= content['year'].apply(lambda x: 2024)
content['month']= content['month'].apply(lambda x: 1)

Seguidamente creamos una columna con la fecha en formato `datetime` e igualmente creamos otra columna con la hora en formato `datetime` a través de la función `.to_datetime()`

In [ ]:
content['date'] = pd.to_datetime(content[['year', 'month', 'day']])
content['hour'] = pd.to_datetime(content['hour'], format='%H')

Para poder crear la columna definitiva con la fecha y la hora, convertimos ambas columnas en `string` a través de la función de datetime (dt) para tal efecto, `.strftime()`

In [ ]:
content['hour'] = content['hour'].dt.strftime('%H:%M:%S')
content['date'] = content['date'].dt.strftime('%m-%d-%y')

Y concatenamos el contenido con el símbolo de la suma y lo reconvertimos a formato `datetime` tal y como vimos anteriormente.

In [ ]:
content['date'] = pd.to_datetime(content['date'] + ' ' + content['hour'])
content['date']

Aquí observamos cómo el dataframe tiene ahora una columna en formato de fecha estándar tanto para Pandas como para Python, sobre el cual ya podemos operar.

## 2. Cálculo de las Métricas

### 2.1 Suma del total de víctimas en cada hora del día

El número total de víctimas es el más grande dentro de toda la estadística de víctimas (fallecidos, heridos graves y heridos leves), y por cantidad es el que nos va a revelar de forma más precisa los patrones de los accidentes de tráfico en el país.  
Podemos por lo tanto calcular el número total de víctimas que se han producido en cada una de las horas del día. Como ya hemos fijado el año y el mes, sólo queda calcular la suma sobre cada una de las horas. Para ello utilizamos la funcion .duplicate() de pandas, que nos permite identificar aquellos valores de la fecha duplicadas sobre las cuales operará la suma.

In [ ]:
content_dups = content[content.duplicated(['date'], keep=False)].copy()

El cálculo se realiza ahora sobre el número total de víctimas computadas a las 24h, y adjuntamos al nuevo dataframe las columnas correspondientes víctimas totales a 24h.

In [ ]:
content_sum = content_dups.groupby('date')['TOTAL_VICTIMAS_24H'].apply(lambda x : x.sum())
content_sum = content_sum.reset_index()
content_sum

De esta forma tenemos la suma del total de víctimas de todas aquellas entradas de fecha que coinciden, realizando así la suma por hora y día de la semana.